In [ ]:
import importlib.util
from pathlib import Path
import subprocess
import sys
import time

RUN_ALL_START = time.perf_counter()

packages_to_check = {
    "optuna": "optuna", "tqdm": "tqdm", "gdown": "gdown",
    "catboost": "catboost", "scikit-learn": "sklearn",
    "hyperopt": "hyperopt", "joblib": "joblib",
}
missing_packages = [name for name, module in packages_to_check.items() if importlib.util.find_spec(module) is None]
for package in missing_packages:
    print(f"Installing {package}...")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package], check=True)

import numpy as np
import optuna
import pandas as pd
from catboost import CatBoostClassifier
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll.base import scope
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import joblib

optuna.logging.set_verbosity(optuna.logging.WARNING)


In [ ]:
## 1.2 Settings
RANDOM_STATE = 42       # the same number everywhere, so results are reproducible
N_SPLITS = 5            # 5-fold cross-validation
TEST_SIZE = 0.2         # 20% of the data is held back and never used for tuning
SCORING = "f1_macro"    # our comparison metric — see section 3 for why
MAX_EVALS = 10          # how many settings Hyperopt tries per model

# StratifiedKFold keeps the class proportions the same in every fold.
# That matters here because our classes are imbalanced.
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# --- The five OCEAN traits (column names start with their letter) ---------
TRAITS = {
    "O": "Openness to Experience",
    "C": "Conscientiousness",
    "E": "Extraversion",
    "A": "Agreeableness",
    "N": "Neuroticism"
}

ocean_order = list(TRAITS.values())

# --- One fixed colour per personality type, used in every plot -------------
TARGET_COLORS = {
    "Resilient":        "#e07a5f",  # muted terracotta/coral
    "Overcontroller":   "#81b29a",  # sage/mint green
    "Undercontroller":  "#f2cc8f",  # warm ochre yellow
    "Moderate":         "#2b5c6f"   # petroleum blue (same as the primary theme colour)
}

target_order = list(TARGET_COLORS.keys())

# --- Colours for titles, info boxes and heatmaps -----------------------------
THEME = {
    "primary":          "#2b5c6f",  # Main petroleum blue for titles & headers
    "success_bg":       "#edf7f4",  # Soft mint/sage background
    "success_border":   "#81b29a",  # Sage/mint green border (matches Overcontroller)
    "danger_bg":        "#fdf2f2",  # Soft coral/red background
    "danger_border":    "#e07a5f",  # Terracotta/coral border (matches Resilient)
    "palette":          "Blues"
}

In [ ]:
# 1.3 Load data
DATA_FILE = Path("DATA") / "data_clean.csv"
df = pd.read_csv(DATA_FILE)

X = df.drop(columns="target")   # features: 19 answers + age + gender + hand
y = df["target"]                # what we predict, kept as readable text

# the 19 questionnaire columns = everything except the demographics and the target
item_cols = [c for c in df.columns if c not in ["age", "gender", "hand", "target"]]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,                 # same class mix in both halves
)

print(f"Rows: {len(df):,} | Train: {len(X_train):,} | Test: {len(X_test):,} | Features: {X.shape[1]}")


In [ ]:
# 2.1 Column definitions
question_cols = [
    'N1', 'N2', 'N3', 'N4', 'N5', 'N6', 'N7', 'N8', 'N9', 'N10',
    'E1', 'E3', 'E4', 'E5', 'E7', 'E9', 'E10', 'C4', 'A4'
]
numeric_cols = question_cols + ['age']
categorical_cols = ['gender', 'hand']

# 2.2 Preprocessing
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
]) 

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
]) 

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

display(preprocessor)

In [ ]:
# 3.1. Pipeline for Logistic Regression
pipe_logreg = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

# 3.2. Pipeline for KNN
pipe_knn = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", KNeighborsClassifier()),
    ]
)

# 3.3. Pipeline for Random Forest
pipe_rf = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(random_state=42)),
    ]
)

# 3.4a. Pipeline for Hist Gradient Boosting
pipe_hgb = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", HistGradientBoostingClassifier(random_state=42)),
    ]
)

# 3.4b. Pipeline for Hist Gradient Boosting (Tuned variant)
pipe_hgb_tuned = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", HistGradientBoostingClassifier(random_state=42)),
    ]
)

# 3.5. Pipeline for CatBoost
pipe_cat = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", CatBoostClassifier(random_state=42, verbose=0)),
    ]
)



In [ ]:
# Baseline comparison: all five models
base_pipelines = {
    "Logistic Regression": pipe_logreg,
    "K-Nearest Neighbors": pipe_knn,
    "Random Forest": pipe_rf,
    "HistGradientBoosting": pipe_hgb,
    "CatBoost": pipe_cat,
}
temp_runs = []
for name, pipeline in base_pipelines.items():
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring=SCORING, n_jobs=-1)
    fitted = clone(pipeline).fit(X_train, y_train)
    temp_runs.append({"name": name, "model": fitted, "cv_score": scores.mean(), "std_cv": scores.std()})

print(pd.DataFrame(temp_runs)[["name", "cv_score", "std_cv"]].sort_values("cv_score", ascending=False))


In [ ]:
# 5.1 Dictionary for GridSearch Tuning & Models Configuration
models_config = {
    "Logistic Regression": {
        "pipeline": pipe_logreg,
        "params": {
            "classifier__C": [0.01, 0.1, 1, 10],
            "classifier__solver": ["lbfgs"],
        },
    },
    "KNN": {
        "pipeline": pipe_knn,
        "params": {
            "classifier__n_neighbors": [3, 5, 7, 9],
            "classifier__weights": ["uniform", "distance"],
        },
    },
    "Random Forest": {
        "pipeline": pipe_rf,
        "params": {
            "classifier__n_estimators": [100, 200],
            "classifier__max_depth": [None, 10, 20],
            "classifier__min_samples_split": [2, 5],
        },
    },
    "HistGradient Boosting": {
        "pipeline": pipe_hgb,
        "params": {
            "classifier__learning_rate": [0.01, 0.05, 0.1],
            "classifier__max_iter": [100, 200],
        },
    },
    "HistGradient Boosting Tuned": {
        "pipeline": pipe_hgb_tuned,
        "params": {
            "classifier__max_leaf_nodes": [31, 32],
            "classifier__min_samples_leaf": [20, 60],
            "classifier__class_weight": ["balanced"],
        },
    },
    # A large CatBoost grid (iterations x depth x learning_rate x l2_leaf_reg = 81 combinations)
    # was too slow, so CatBoost gets a small grid: tree depth and class weights.
    # Two parts, because CatBoost does not accept auto_class_weights=None:
    # the first part trains without class weights, the second part with them.
    "CatBoost": {
        "pipeline": pipe_cat,
        "params": [
            {"classifier__depth": [4, 6]},
            {"classifier__depth": [4, 6],
             "classifier__auto_class_weights": ["SqrtBalanced", "Balanced"]},
        ],
    },
}

In [ ]:
# Grid search
temp_grid_runs = []
for name, config in models_config.items():
    search = GridSearchCV(
        config["pipeline"], config["params"], cv=cv, scoring=SCORING,
        n_jobs=-1, error_score="raise"
    )
    search.fit(X_train, y_train)
    temp_grid_runs.append({
        "name": name,
        "model": search.best_estimator_,
        "mean_cv": search.best_score_,
        "best_params": search.best_params_,
    })

print(pd.DataFrame(temp_grid_runs)[["name", "mean_cv", "best_params"]].sort_values("mean_cv", ascending=False))


In [ ]:
# 6.2 Dictionary for Hyperopt Tuning & Models Configuration and Spaces
hyperopt_configs = {
    "Logistic Regression": {
        "pipeline": pipe_logreg,
        "space": {
            "classifier__C": hp.loguniform("classifier__C", np.log(0.01), np.log(10))
        }
    },
    "K-Nearest Neighbors": {
        "pipeline": pipe_knn,
        "space": {
            # every whole number from 3 to 11 (a step of 2 would only give 4, 6, 8 and 10)
            "classifier__n_neighbors": scope.int(hp.quniform("classifier__n_neighbors", 3, 11, 1))
        }
    },
    "Random Forest": {
        "pipeline": pipe_rf,
        "space": {
            "classifier__n_estimators": scope.int(hp.quniform("classifier__n_estimators", 50, 300, 50)),
            "classifier__max_depth": scope.int(hp.quniform("classifier__max_depth", 5, 30, 5)),
            "classifier__min_samples_split": scope.int(hp.quniform("classifier__min_samples_split", 2, 10, 2))
        }
    },
    "HistGradientBoosting": {
        "pipeline": pipe_hgb,
        "space": {
            "classifier__learning_rate": hp.uniform("classifier__learning_rate", 0.02, 0.3),
            "classifier__max_iter": scope.int(hp.quniform("classifier__max_iter", 50, 250, 50))
        }
    }
}


In [ ]:
# Hyperopt search
temp_hyper_runs = []
for name, config in hyperopt_configs.items():
    pipeline = clone(config["pipeline"])

    def objective(params):
        candidate = clone(pipeline).set_params(**params)
        score = cross_val_score(candidate, X_train, y_train, cv=cv, scoring=SCORING, n_jobs=-1).mean()
        return {"loss": -score, "status": STATUS_OK}

    best = fmin(
        fn=objective, space=config["space"], algo=tpe.suggest,
        max_evals=MAX_EVALS, trials=Trials(), rstate=np.random.default_rng(RANDOM_STATE), verbose=0,
    )
    best_params = {
        key: int(value) if any(token in key for token in ["n_estimators", "max_depth", "min_samples_split", "n_neighbors", "max_iter"]) else value
        for key, value in best.items()
    }
    fitted = clone(pipeline).set_params(**best_params).fit(X_train, y_train)
    score = cross_val_score(fitted, X_train, y_train, cv=cv, scoring=SCORING, n_jobs=-1).mean()
    temp_hyper_runs.append({"name": name, "model": fitted, "mean_cv": score, "best_params": best_params})

print(pd.DataFrame(temp_hyper_runs)[["name", "mean_cv", "best_params"]].sort_values("mean_cv", ascending=False))


In [ ]:
# Select the champion by cross-validated macro F1
def result_rows(runs, tuning, score_key):
    return [
        {"model": run["name"], "tuning": tuning, "pipeline": run["model"],
         "cv_f1_macro": run[score_key], "params": run.get("best_params", {})}
        for run in runs
    ]

all_runs = pd.DataFrame(
    result_rows(temp_runs, "none", "cv_score")
    + result_rows(temp_grid_runs, "grid search", "mean_cv")
    + result_rows(temp_hyper_runs, "hyperopt", "mean_cv")
).sort_values("cv_f1_macro", ascending=False).reset_index(drop=True)

champion_row = all_runs.iloc[0]
CHAMPION_NAME = champion_row["model"]
CHAMPION_TUNING = champion_row["tuning"]
CHAMPION_CV = champion_row["cv_f1_macro"]
champion_params = champion_row["params"]
champion_pipeline = clone(champion_row["pipeline"]).fit(X_train, y_train)

y_pred_test = np.ravel(champion_pipeline.predict(X_test))
CHAMPION_TEST = f1_score(y_test, y_pred_test, average="macro")
champion_test_acc = accuracy_score(y_test, y_pred_test)

print(all_runs[["model", "tuning", "cv_f1_macro"]])
print(f"Champion: {CHAMPION_NAME} ({CHAMPION_TUNING}) | CV macro F1: {CHAMPION_CV:.3f} | Test macro F1: {CHAMPION_TEST:.3f}")


In [ ]:
# Held-out test-set evaluation
report = classification_report(y_test, y_pred_test, labels=target_order, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report).T.loc[target_order + ["macro avg", "weighted avg"]]
print(report_df[["precision", "recall", "f1-score", "support"]])
print(f"Test accuracy: {champion_test_acc:.1%}")


In [ ]:
# Refit the champion on all data and save the complete pipeline
best_pipeline = clone(champion_pipeline).fit(X, y)
model_path = Path("models") / "personality_pipeline.joblib"
model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(best_pipeline, model_path)
print(f"Saved: {model_path}")


In [ ]:
# Total runtime
total_seconds = time.perf_counter() - RUN_ALL_START
minutes, seconds = divmod(int(total_seconds), 60)
print(f"Notebook runtime: {minutes} min {seconds} sec")
